# Phase 3: Advanced Python for AI Systems
Covers Decorators & Closures, Static Type Hinting & Pydantic Validation, Concurrency (`asyncio`, `threading`, `multiprocessing`), Subprocess Management, Memory Management (`copy`), Dynamic Code Execution (`eval`, `exec`, `compile`), Packaging, and Testing.

## 1. Decorators, Closures & `functools.wraps`

In [ ]:
import time
from functools import wraps

# Parameterized Decorator for LLM API Retry & Latency Tracking
def retry_with_Metrics(max_retries: int = 3):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_retries + 1):
                try:
                    start = time.perf_counter()
                    result = func(*args, **kwargs)
                    ms = (time.perf_counter() - start) * 1000
                    print(f"[Attempt {attempt}] {func.__name__} succeeded in {ms:.2f}ms")
                    return result
                except Exception as e:
                    print(f"[Attempt {attempt}] Failed: {e}")
                    if attempt == max_retries:
                        raise
        return wrapper
    return decorator

@retry_with_Metrics(max_retries=2)
def call_llm_endpoint(prompt: str) -> dict:
    return {"response": f"Processed: {prompt}", "tokens": len(prompt.split())}

print(call_llm_endpoint("Explain self-attention in Transformers"))

## 2. Static Type Hinting (`typing`) & Schema Validation (`Pydantic`)

In [ ]:
from typing import Optional, Union, TypedDict, Protocol, Annotated

# 1. TypedDict for lightweight dictionary structure typing
class AgentState(TypedDict):
    messages: list[str]
    step_count: int
    active_tool: Optional[str]

# 2. Protocol for Structural Subtyping (Duck Typing interface for Vector Stores)
class VectorStoreProtocol(Protocol):
    def similarity_search(self, query: str, k: int) -> list[str]:
        ...

class MockQdrantStore:
    def similarity_search(self, query: str, k: int = 3) -> list[str]:
        return [f"Doc {i} matching '{query}'" for i in range(1, k + 1)]

def run_rag_retrieval(store: VectorStoreProtocol, query: str) -> list[str]:
    return store.similarity_search(query, k=2)

state: AgentState = {"messages": ["Hello"], "step_count": 1, "active_tool": "search"}
print("Agent State:", state)
print("Retrieved Docs:", run_rag_retrieval(MockQdrantStore(), "LoRA fine-tuning"))

# 3. Pydantic for Runtime LLM Structured Output Validation (if installed)
try:
    from pydantic import BaseModel, Field

    class ExtractedEntity(BaseModel):
        entity_name: str = Field(description="Name of the extracted entity")
        confidence: float = Field(ge=0.0, le=1.0)

    item = ExtractedEntity(entity_name="PyTorch", confidence=0.98)
    print("Pydantic JSON Schema:", item.model_dump_json())
except ImportError:
    print("Pydantic not installed in current env; run: pip install pydantic")

## 3. Concurrency & Parallelism (`asyncio`, `threading`, `multiprocessing`)

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

# 1. Asyncio for high-concurrency LLM API calls
async def fetch_llm_completion(prompt_id: int) -> dict:
    await asyncio.sleep(0.05)  # Simulate non-blocking network I/O
    return {"id": prompt_id, "status": "completed"}

async def batch_process_prompts():
    tasks = [fetch_llm_completion(i) for i in range(1, 5)]
    results = await asyncio.gather(*tasks)
    return results

# Note: In Jupyter, top-level 'await' works directly
async_results = await batch_process_prompts()
print("Async Batch Results:", async_results)

# 2. Threading via ThreadPoolExecutor for synchronous I/O tasks
def load_file_chunk(chunk_id: int) -> str:
    return f"chunk_{chunk_id}_loaded"

with ThreadPoolExecutor(max_workers=4) as executor:
    chunks = list(executor.map(load_file_chunk, range(4)))
print("Threaded Chunks:", chunks)

## 4. Subprocess Management (`subprocess`, `sys`)

In [ ]:
import subprocess
import sys

# Running external CLI commands safely (e.g., checking Python version, git, or nvidia-smi)
cmd_result = subprocess.run(
    [sys.executable, "--version"],
    capture_output=True,
    text=True,
    check=True
)
print("Subprocess Output:", cmd_result.stdout.strip())

## 5. Memory Management (`copy`, Shallow vs. Deep Copy, `id`)

In [ ]:
import copy

# Crucial when saving best model state_dicts or cloning nested agent states
model_state = {"layer1": {"weights": [0.1, 0.2, 0.3]}, "epoch": 1}

shallow_clone = copy.copy(model_state)
deep_clone = copy.deepcopy(model_state)

# Mutating nested list in original state
model_state["layer1"]["weights"][0] = 99.9

print("Original Weights:     ", model_state["layer1"]["weights"])
print("Shallow Clone (Leaked):", shallow_clone["layer1"]["weights"])
print("Deep Clone (Safe):    ", deep_clone["layer1"]["weights"])

## 6. Dynamic Code Execution & Compilation (`eval`, `exec`, `compile`, `__import__`)

In [ ]:
# Used inside AI Coding Agents / Sandboxed Python Interpreters
math_Module = __import__("math")

# 1. Restricted eval() for safe mathematical expression evaluation
allowed_globals = {"__builtins__": {}, "sqrt": math_Module.sqrt}
calc_result = eval("sqrt(144) + 10", allowed_globals, {})
print("Sandboxed eval() result:", calc_result)

# 2. compile() + exec() for executing LLM-generated Python snippets
llm_generated_code = """
def compute_f1(precision, recall):
    return 2 * (precision * recall) / (precision + recall)
metric_output = round(compute_f1(0.85, 0.90), 4)
"""
local_namespace = {}
compiled_code = compile(llm_generated_code, filename="<llm_sandbox>", mode="exec")
exec(compiled_code, {"__builtins__": {"round": round}}, local_namespace)
print("Executed LLM Code Output (F1 Score):", local_namespace["metric_output"])

## 7. Packaging Structure & Automated Testing (`unittest` / `pytest` patterns)

In [ ]:
import unittest
from unittest.mock import MagicMock

# Mocking expensive LLM calls during unit testing
class LLMService:
    def generate(self, prompt: str) -> str:
        raise RuntimeError("Should not hit real API in unit tests!")

def summarize_text(service: LLMService, text: str) -> str:
    if not text.strip():
        raise ValueError("Input text cannot be empty")
    return service.generate(f"Summarize: {text}")

class TestAIPipeline(unittest.TestCase):
    def test_summarize_with_mock(self):
        mock_service = LLMService()
        mock_service.generate = MagicMock(return_value="Mocked Summary")

        result = summarize_text(mock_service, "Long article about AI agents.")
        self.assertEqual(result, "Mocked Summary")
        mock_service.generate.assert_called_once_with("Summarize: Long article about AI agents.")

    def test_empty_input_raises(self):
        with self.assertRaises(ValueError):
            summarize_text(LLMService(), "   ")

# Run unit tests directly inside the Jupyter Notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestAIPipeline)
unittest.TextTestRunner(verbosity=2).run(suite)